# Waymaker Artemis — Mission 005-D: 개발 후보 20건 실제 추론

무료 Colab T4에서 저장소 코드(커밋 `6dfe161165852922db592e6a1a76578c07e7fada`)로 공식 `google/gemma-4-E4B-it`(revision `ee0ef60…`)를 NF4로 **한 번** 올리고,
개발 후보 20건의 응답을 생성합니다. 먼저 DEV-001~003 파일럿, 이어서 같은 세션에서 DEV-004~020.

- 이 노트북은 **생성만** 합니다. 채점은 저장소에서 `score_dev20.py`로 따로 합니다. 정답·채점 기준은 프롬프트에 들어가지 않습니다.
- held-out 40건은 실행하지 않습니다. 결과는 개발 후보 진단 수치이며 Official Gold Accuracy는 `N/A`입니다.
- 비용: 무료 런타임만 씁니다. 유료 GPU, 컴퓨팅 단위 구매, Google Cloud 결제는 쓰지 않습니다. 토큰이 필요 없습니다.
- 각 사례는 끝날 때마다 `generations.jsonl`에 저장됩니다. 연결이 끊기면 같은 셀을 다시 실행하면 남은 사례만 생성합니다.

In [ ]:
# 1. GPU·VRAM·디스크 확인 (읽기 전용)
import shutil, subprocess, sys
print(sys.version)
print(subprocess.run(['nvidia-smi', '--query-gpu=name,memory.total,memory.free,driver_version',
                      '--format=csv,noheader'], capture_output=True, text=True).stdout)
print('Disk free GB:', round(shutil.disk_usage('/content').free / 1e9, 2))

## 2. 무료 실행 환경 확인

1. **런타임 → 런타임 유형 변경**에서 `T4 GPU`인지 봅니다.
2. **리소스 보기**에서 구독 없음·컴퓨팅 단위 미사용 상태인지 봅니다.
3. 둘 다 맞으면 아래 값을 `True`로 바꿉니다.

In [ ]:
FREE_TIER_CONFIRMED = False  # 위 두 가지를 직접 확인한 뒤에만 True로 바꿉니다
assert FREE_TIER_CONFIRMED, '무료 T4 런타임인지 먼저 확인하세요'
gpu = subprocess.run(['nvidia-smi', '--query-gpu=name', '--format=csv,noheader'], capture_output=True, text=True).stdout.strip()
assert gpu == 'Tesla T4', f'예상한 무료 T4가 아닙니다: {gpu!r}'

In [ ]:
# 3. 코드와 패키지 버전 고정
CODE_COMMIT = '6dfe161165852922db592e6a1a76578c07e7fada'
!git init -q /content/Visable
!git -C /content/Visable remote add origin https://github.com/club-paradiso/Visable.git
# artemis_eval -> procedure_route imports backend/services (standard library only, about 5MB).
!git -C /content/Visable sparse-checkout set experiments/waymaker-ai backend/services
!git -C /content/Visable fetch -q --depth 1 --filter=blob:none origin {CODE_COMMIT}
!git -C /content/Visable checkout -q FETCH_HEAD
head = subprocess.run(['git', '-C', '/content/Visable', 'rev-parse', 'HEAD'], capture_output=True, text=True).stdout.strip()
assert head == CODE_COMMIT, f'코드 커밋 불일치: {head}'
%pip install -q "transformers==5.20.2" "bitsandbytes==0.50.2"
import importlib.metadata as md
print({n: md.version(n) for n in ('torch', 'transformers', 'bitsandbytes', 'accelerate', 'huggingface-hub', 'tokenizers')})

In [ ]:
# 4. 체크포인트 다운로드(이 셀만 가중치를 받습니다, 약 16GB)와 SHA-256 확인
import json, os
REPO = '/content/Visable/experiments/waymaker-ai'
sys.path[:0] = [REPO, REPO + '/artemis_bootstrap']
policy = json.load(open(REPO + '/configs/artemis_cuda_runtime.json'))
MODEL, REVISION = policy['model']['checkpoint'], policy['model']['revision']
assert (MODEL, REVISION) == ('google/gemma-4-E4B-it', 'ee0ef6023621cff504d758262d4e04895a5af4a2')
assert shutil.disk_usage('/content').free / 1e9 >= policy['limits']['min_free_disk_gb_for_download']
from huggingface_hub import snapshot_download
from pathlib import Path
snapshot = Path(snapshot_download(MODEL, revision=REVISION, token=False))
from waymaker_ai import cuda_policy
verified = cuda_policy.validate_snapshot(snapshot)  # 1~2분
print(verified['weight_integrity'], verified['artifact_integrity']['model.safetensors']['digest'])

In [ ]:
# 5. CUDA 사전 검사. 이후로는 네트워크를 쓰지 않습니다.
os.environ['HF_HUB_OFFLINE'] = '1'
os.environ['TRANSFORMERS_OFFLINE'] = '1'
import gemma4_cuda_runtime as cuda
check = cuda.preflight()
print('blockers:', check['blockers'], '| version drift:', check['probe']['version_drift'])

In [ ]:
# 6. 모델을 한 번만 올리고 NF4 검증
prepared = cuda.PreparedRun()
print('loaded:', prepared.prepare(), '| blocker:', prepared.blocker)
quant = prepared.header.get('load', {}).get('quantization', {})
print({k: quant.get(k) for k in ('passed', 'quantized_decoder_layers', 'linear4bit_modules', 'quant_types', 'parameter_devices')})

In [ ]:
# 7. 파일럿 DEV-001~003 (채점하지 않고 실행 상태만 봅니다)
import run_dev20_cuda
OUT_DIR = '/content/artemis_mission005d_dev20'
print(run_dev20_cuda.main(['--out-dir', OUT_DIR, '--cases', 'pilot'], prepared=prepared))
for line in open(OUT_DIR + '/generations.jsonl'):
    g = json.loads(line)
    print(g['case_id'], g['execution_status'], g['generated_token_count'], g['finish_reason'], g['metrics'] and round(g['metrics']['wall_seconds'], 1))

In [ ]:
# 8. 같은 세션에서 DEV-004~020. 연결이 끊겼다면 1~6을 다시 실행한 뒤 이 셀을 실행합니다(이미 저장된 사례는 건너뜁니다).
print(run_dev20_cuda.main(['--out-dir', OUT_DIR, '--cases', 'rest'], prepared=prepared))

In [ ]:
# 9. 결과 해시와 내려받기
import hashlib
for name in ('generations.jsonl', 'sessions.jsonl'):
    raw = open(f'{OUT_DIR}/{name}', 'rb').read()
    print(name, len(raw), hashlib.sha256(raw).hexdigest())
!cd /content && zip -qr artemis_mission005d_dev20.zip artemis_mission005d_dev20
from google.colab import files
files.download('/content/artemis_mission005d_dev20.zip')

## 끝난 뒤

- 내려받은 zip의 `generations.jsonl`, `sessions.jsonl`을 고치지 않고 `experiments/waymaker-ai/artemis_bootstrap/runs/mission005d-dev20/`에 넣고
  `python artemis_bootstrap/score_dev20.py --out-dir artemis_bootstrap/runs/mission005d-dev20`으로 채점합니다.
- **런타임 → 런타임 연결 해제 및 삭제**로 GPU를 반납합니다.